In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.pipeline_monitoring (
  run_date DATE,
  layer STRING,
  target_table STRING,
  status STRING,
  message STRING,
  logged_at TIMESTAMP
)
USING DELTA
""")


In [0]:
MONITOR_TABLE = "workspace.default.pipeline_monitoring"

def log_run(run_date: str, layer: str, target_table: str, status: str, message: str = ""):
    (spark.createDataFrame([(run_date, layer, target_table, status, message)],
                           "run_date string, layer string, target_table string, status string, message string")
         .selectExpr(
             "cast(run_date as date) as run_date",
             "layer",
             "target_table",
             "status",
             "message",
             "current_timestamp() as logged_at"
         )
         .write.format("delta")
         .mode("append")
         .saveAsTable(MONITOR_TABLE)
    )
